# 03 · Off-the-shelf baseline: zero-shot multilingual ASR

**Question.** Before training anything on DhoNam, what do existing public models give a Dholuo speaker?

* **MMS-1b-all** (Meta, Massively Multilingual Speech) ships per-language adapters. This notebook
  checks *programmatically* whether a `luo` adapter exists in the downloaded checkpoint; it does not
  assume Dholuo support. If one exists, it is evaluated zero-shot on the official validation and test
  sets. Its training data and orthography differ from DhoNam, so CER is as informative as WER here.
* A trivial reference point: an empty hypothesis scores WER = 100 %.

Kaggle: **GPU T4 x1** (or P100), Internet ON. Input: `splits.csv` from notebook 02.

In [ ]:
# ── Setup: locate the project source (src/dholuo_asr) and the output folder ──────────
import os, sys, subprocess, json, warnings
from pathlib import Path
warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", category=UserWarning)

# After pushing the repo to GitHub, put its URL here (Kaggle: Settings → Internet ON).
# Alternatively upload the repo as a private Kaggle dataset and attach it as an input.
REPO_URL = os.environ.get("DHOLUO_REPO_URL", "https://github.com/YOUR-USERNAME/dholuo-asr.git")

def _find_src(max_depth=4):
    for root in [Path.cwd(), Path.cwd().parent, Path("/kaggle/working"), Path("/kaggle/input")]:
        if not root.exists():
            continue
        base = len(root.parts)
        for dirpath, dirnames, filenames in os.walk(root):
            if len(Path(dirpath).parts) - base >= max_depth or len(filenames) > 500:
                dirnames[:] = []          # don't descend into audio folders
            if Path(dirpath).name == "dholuo_asr" and "__init__.py" in filenames:
                return Path(dirpath).parent
    return None

SRC = _find_src()
if SRC is None and "YOUR-USERNAME" not in REPO_URL:
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL, "/kaggle/working/dholuo-asr"], check=True)
    SRC = Path("/kaggle/working/dholuo-asr/src")
if SRC is None:
    raise RuntimeError("Project source not found: set REPO_URL (Internet ON) or attach the repo as a dataset.")
sys.path.insert(0, str(SRC))
REPO = SRC.parent
ON_KAGGLE = Path("/kaggle/working").exists()
OUT = Path(os.environ.get("DHOLUO_OUT", "/kaggle/working" if ON_KAGGLE else REPO / "outputs"))
OUT.mkdir(parents=True, exist_ok=True)
ARTIFACT_ROOTS = [OUT, Path("/kaggle/input")]
print("source :", SRC)
print("outputs:", OUT)

In [ ]:
import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
pd.set_option("display.max_columns", 50); pd.set_option("display.width", 160)
plt.rcParams.update({"figure.dpi": 110, "axes.spines.top": False, "axes.spines.right": False,
                     "axes.grid": True, "grid.alpha": 0.3, "font.size": 10})
DIALECT_COLORS = {"Milambo": "#2a6f97", "Nyanduat": "#e07a5f"}
FIG = OUT / "figures"; FIG.mkdir(exist_ok=True)
def savefig(name):
    plt.tight_layout(); plt.savefig(FIG / f"{name}.png", bbox_inches="tight"); plt.show()

In [ ]:
import torch
from dholuo_asr.utils import find_artifact, dump_json, env_info
from dholuo_asr.data import ASRDataset
from dholuo_asr.metrics import score_frame, corpus_rates, speaker_bootstrap, group_rates
splits = pd.read_csv(find_artifact("splits.csv", ARTIFACT_ROOTS), keep_default_na=False, na_values=[""])
MMS_CKPT = "facebook/mms-1b-all"
LANG = "luo"     # ISO 639-3 code for Dholuo
RUN = OUT / "results" / "mms_zeroshot"; RUN.mkdir(parents=True, exist_ok=True)
dev = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", dev)

In [ ]:
from transformers import AutoProcessor, Wav2Vec2ForCTC
status = {"checkpoint": MMS_CKPT, "lang": LANG}
try:
    processor = AutoProcessor.from_pretrained(MMS_CKPT)
    langs = getattr(processor.tokenizer, "vocab", {})
    status["n_languages_in_tokenizer"] = len(langs) if isinstance(langs, dict) else None
    try:                                   # decisive test: can the tokenizer switch to Dholuo?
        processor.tokenizer.set_target_lang(LANG)
        status["adapter_available"] = True
    except (ValueError, KeyError) as e:
        status["adapter_available"] = False
        status["reason"] = str(e)[:300]
    print(f"'{LANG}' vocabulary available in {MMS_CKPT}: {status['adapter_available']}")
except Exception as e:
    status["error"] = repr(e)
    status["adapter_available"] = False
    print("Could not load the MMS processor (Internet off?):", e)
dump_json(status, RUN / "status.json")

In [ ]:
if status.get("adapter_available"):
    model = Wav2Vec2ForCTC.from_pretrained(MMS_CKPT, target_lang=LANG, ignore_mismatched_sizes=True).eval()
    model = (model.half() if dev == "cuda" else model).to(dev)   # loads the 'luo' adapter weights

    def transcribe(df, max_batch_seconds=240):
        ds = ASRDataset(df, "text_raw", cache_dir="/tmp/audio_cache")
        from dholuo_asr.data import DurationBatchSampler
        hyps = {}
        for batch in DurationBatchSampler(ds.durations, max_batch_seconds, shuffle=False):
            wavs = [ds[i]["wav"] for i in batch]
            inp = processor(wavs, sampling_rate=16000, return_tensors="pt", padding=True)
            with torch.no_grad():
                am = inp["attention_mask"].to(dev) if "attention_mask" in inp else None
                logits = model(inp.input_values.to(dev, model.dtype), attention_mask=am).logits
            ids = logits.argmax(-1).cpu()
            for i, txt in zip(batch, processor.batch_decode(ids)):
                hyps[i] = txt
        out = ds.df.copy(); out["hyp"] = [hyps[i] for i in range(len(out))]; out["ref"] = out["text_raw"]
        return out

    metrics = {"name": "mms_zeroshot", "status": status, "env": env_info()}
    for split in ["val", "test"]:
        p = transcribe(splits[splits.split == split])
        p.to_csv(RUN / f"predictions_{split}.csv", index=False)
        s = score_frame(p)
        metrics[split] = {**corpus_rates(s), **speaker_bootstrap(s)}
        metrics[split]["by_dialect"] = group_rates(s, "dialect")[["wer", "cer", "n_utts"]].to_dict("index")
        print(split, {k: round(v, 4) for k, v in corpus_rates(s).items() if isinstance(v, float)})
        display(p[["ref", "hyp"]].head(8))
    dump_json(metrics, RUN / "metrics.json")
elif "error" in status:
    print("The checkpoint could not be downloaded, so Dholuo support is UNKNOWN (not 'unsupported'). "
          "Turn Internet ON and re-run before reporting anything about MMS zero-shot.")
else:
    print("No Dholuo adapter in this checkpoint: report zero-shot MMS as 'not supported'; the baseline "
          "ladder then starts at the from-scratch BiLSTM-CTC (notebook 04).")

**Interpretation checklist (fill in after running):**
* Is the zero-shot output in the same orthography as DhoNam (apostrophes, spelling conventions)? Compare CER vs WER.
* Does performance differ between Milambo and Nyanduat speakers?
* This number is the *starting point* that fine-tuning must beat to justify its compute.